# teta-ml-2026 | Alternative baseline (LightGBM/CatBoost)

## Intro
- Делаем компактный pipeline: короткая EDA -> feature engineering -> кодирование -> CV-обучение -> submission.
- Основной алгоритм: **LightGBMRegressor** (быстрый бустинг на табличных данных), fallback: **CatBoostRegressor** если `lightgbm` недоступен.
- В признаках делаем упор на: географию (дистанции/кластеры), дату последнего отзыва, текст `name` (TF-IDF + SVD), групповые и host-агрегации без утечки target.


## Загрузка библиотек и данных

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.cluster import KMeans
from sklearn.preprocessing import OrdinalEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

SEED = 42
DATA_DIR = '/kaggle/input/teta-ml-2026'

train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')
sample_sub = pd.read_csv(f'{DATA_DIR}/sample_submition.csv')

target_col = 'target'
id_col = sample_sub.columns[0]

print('Train shape:', train.shape)
print('Test shape:', test.shape)
print('Sample submission shape:', sample_sub.shape)

## Краткая EDA

In [ ]:
display(train.head(2))
display(test.head(2))

missing_train = train.isna().mean().sort_values(ascending=False).head(15)
missing_test = test.isna().mean().sort_values(ascending=False).head(15)
print('Top missing columns in train:')
display(missing_train)
print('Top missing columns in test:')
display(missing_test)

print('Target describe:')
display(train[target_col].describe())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

sns.histplot(train[target_col], bins=60, kde=True, ax=axes[0])
axes[0].set_title('Target distribution')

group_col = 'type_house' if 'type_house' in train.columns else 'location_cluster'
tmp_group = train.groupby(group_col, as_index=False)[target_col].mean().sort_values(target_col, ascending=False).head(15)
sns.barplot(data=tmp_group, x=target_col, y=group_col, ax=axes[1])
axes[1].set_title(f'Mean target by {group_col}')

if {'lat', 'lon'}.issubset(train.columns):
    sns.scatterplot(data=train.sample(min(8000, len(train)), random_state=SEED), x='lon', y='lat',
                    hue=target_col, palette='viridis', alpha=0.35, s=12, ax=axes[2], legend=False)
    axes[2].set_title('Geo scatter (sample)')
else:
    axes[2].text(0.5, 0.5, 'lat/lon not found', ha='center', va='center')

plt.tight_layout()
plt.show()

## Feature Engineering

In [ ]:
def safe_col(df, col, default=np.nan):
    return df[col] if col in df.columns else pd.Series(default, index=df.index)


def build_text_features(full_df, n_components=15):
    text = safe_col(full_df, 'name', '').fillna('').astype(str)

    text_df = pd.DataFrame(index=full_df.index)
    text_df['name_len'] = text.str.len()
    text_df['name_word_count'] = text.str.split().str.len()
    text_df['name_upper_count'] = text.str.count(r'[A-ZА-Я]')
    text_df['name_digit_count'] = text.str.count(r'\d')
    text_df['name_special_count'] = text.str.count(r'[^\w\s]')

    words = text.str.split()
    text_df['name_avg_word_len'] = words.apply(lambda x: np.mean([len(w) for w in x]) if isinstance(x, list) and len(x) else 0)

    tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=3, strip_accents='unicode')
    X_tfidf = tfidf.fit_transform(text)

    n_comp = min(n_components, X_tfidf.shape[1] - 1) if X_tfidf.shape[1] > 1 else 1
    svd = TruncatedSVD(n_components=n_comp, random_state=SEED)
    X_svd = svd.fit_transform(X_tfidf)

    svd_cols = [f'name_svd_{i}' for i in range(X_svd.shape[1])]
    svd_df = pd.DataFrame(X_svd, columns=svd_cols, index=full_df.index)

    return pd.concat([text_df, svd_df], axis=1)


def add_geo_features(full_df):
    out = pd.DataFrame(index=full_df.index)
    lat = safe_col(full_df, 'lat')
    lon = safe_col(full_df, 'lon')

    points = {
        'times_square': (40.7580, -73.9855),
        'brooklyn': (40.6782, -73.9442),
        'queens': (40.7282, -73.7949),
        'central_park': (40.7829, -73.9654),
    }

    for name, (p_lat, p_lon) in points.items():
        out[f'dist_{name}'] = np.sqrt((lat - p_lat) ** 2 + (lon - p_lon) ** 2)

    dist_cols = [c for c in out.columns if c.startswith('dist_')]
    out['dist_min_city_point'] = out[dist_cols].min(axis=1)

    for d in [1, 2, 3]:
        out[f'lat_round_{d}'] = lat.round(d)
        out[f'lon_round_{d}'] = lon.round(d)

    geo_base = pd.DataFrame({'lat': lat.fillna(lat.median()), 'lon': lon.fillna(lon.median())})
    for k in [10, 25, 50]:
        km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
        out[f'geo_cluster_{k}'] = km.fit_predict(geo_base)

    return out


def add_date_features(full_df):
    out = pd.DataFrame(index=full_df.index)
    dt = pd.to_datetime(safe_col(full_df, 'last_dt'), errors='coerce')

    anchor = dt.max()
    if pd.isna(anchor):
        anchor = pd.Timestamp('2026-01-01')

    out['last_dt_missing'] = dt.isna().astype(int)
    out['days_since_last_review'] = (anchor - dt).dt.days
    out['days_since_last_review'] = out['days_since_last_review'].fillna(out['days_since_last_review'].median())

    month = dt.dt.month.fillna(0)
    dow = dt.dt.dayofweek.fillna(0)
    out['month_sin'] = np.sin(2 * np.pi * month / 12)
    out['month_cos'] = np.cos(2 * np.pi * month / 12)
    out['dayofweek_sin'] = np.sin(2 * np.pi * dow / 7)
    out['dayofweek_cos'] = np.cos(2 * np.pi * dow / 7)

    out['review_recency_bin'] = pd.cut(
        out['days_since_last_review'],
        bins=[-1, 7, 30, 90, 180, 365, 10_000],
        labels=False
    ).astype(float).fillna(-1)

    return out


def add_host_features(full_df):
    out = pd.DataFrame(index=full_df.index)
    host_col = 'host_name'
    if host_col not in full_df.columns:
        return out

    g = full_df.groupby(host_col)
    out['host_name_freq'] = full_df[host_col].map(full_df[host_col].value_counts())

    if 'location' in full_df.columns:
        out['host_nunique_location'] = full_df[host_col].map(g['location'].nunique())
    if 'type_house' in full_df.columns:
        out['host_nunique_type_house'] = full_df[host_col].map(g['type_house'].nunique())

    if 'sum' in full_df.columns:
        out['host_mean_sum'] = full_df[host_col].map(g['sum'].mean())
        out['host_median_sum'] = full_df[host_col].map(g['sum'].median())
    if 'amt_reviews' in full_df.columns:
        out['host_mean_amt_reviews'] = full_df[host_col].map(g['amt_reviews'].mean())

    return out


def add_group_stats(full_df):
    out = pd.DataFrame(index=full_df.index)
    numeric_cols = [c for c in ['sum', 'min_days', 'amt_reviews', 'avg_reviews', 'total_host'] if c in full_df.columns]

    if 'location_cluster' in full_df.columns and 'type_house' in full_df.columns:
        full_df = full_df.copy()
        full_df['location_cluster_type_house'] = full_df['location_cluster'].astype(str) + '_' + full_df['type_house'].astype(str)

    group_cols = [c for c in ['location_cluster', 'location', 'type_house', 'location_cluster_type_house', 'geo_cluster_25'] if c in full_df.columns]

    for gc in group_cols:
        grp = full_df.groupby(gc)
        for nc in numeric_cols:
            out[f'{gc}_{nc}_mean'] = full_df[gc].map(grp[nc].mean())
            out[f'{gc}_{nc}_median'] = full_df[gc].map(grp[nc].median())
            out[f'{gc}_{nc}_std'] = full_df[gc].map(grp[nc].std())

    if {'sum', 'location'}.issubset(full_df.columns):
        loc_mean = full_df.groupby('location')['sum'].mean()
        loc_med = full_df.groupby('location')['sum'].median()
        out['sum_div_location_mean'] = full_df['sum'] / full_df['location'].map(loc_mean).replace(0, np.nan)
        out['sum_minus_location_median'] = full_df['sum'] - full_df['location'].map(loc_med)

    if {'amt_reviews', 'type_house'}.issubset(full_df.columns):
        th_mean = full_df.groupby('type_house')['amt_reviews'].mean()
        out['amt_reviews_div_type_house_mean'] = full_df['amt_reviews'] / full_df['type_house'].map(th_mean).replace(0, np.nan)

    if {'min_days', 'location_cluster'}.issubset(full_df.columns):
        lc_med = full_df.groupby('location_cluster')['min_days'].median()
        out['min_days_minus_location_cluster_median'] = full_df['min_days'] - full_df['location_cluster'].map(lc_med)

    return out


In [ ]:
full = pd.concat([train.drop(columns=[target_col]), test], axis=0, ignore_index=True)

geo_feats = add_geo_features(full)
full = pd.concat([full, geo_feats], axis=1)

date_feats = add_date_features(full)
text_feats = build_text_features(full, n_components=15)
host_feats = add_host_features(full)
group_feats = add_group_stats(full)

full_fe = pd.concat([full, date_feats, text_feats, host_feats, group_feats], axis=1)

for c in full_fe.columns:
    if full_fe[c].dtype == 'object':
        full_fe[c] = full_fe[c].fillna('missing').astype(str)
    else:
        full_fe[c] = full_fe[c].replace([np.inf, -np.inf], np.nan)
        full_fe[c] = full_fe[c].fillna(full_fe[c].median() if pd.api.types.is_numeric_dtype(full_fe[c]) else 'missing')

X_train = full_fe.iloc[:len(train)].copy()
X_test = full_fe.iloc[len(train):].copy()
y = train[target_col].values

print('Feature matrix train:', X_train.shape)
print('Feature matrix test:', X_test.shape)

## Кодирование признаков

In [ ]:
def frequency_encode(train_df, test_df, col):
    freq = pd.concat([train_df[col], test_df[col]], axis=0).value_counts(dropna=False)
    train_df[f'{col}_freq'] = train_df[col].map(freq)
    test_df[f'{col}_freq'] = test_df[col].map(freq)


cat_cols = [c for c in X_train.columns if X_train[c].dtype == 'object']

for c in cat_cols:
    frequency_encode(X_train, X_test, c)

small_cat = []
high_cat = []
for c in cat_cols:
    nun = pd.concat([X_train[c], X_test[c]], axis=0).nunique()
    if nun <= 30:
        small_cat.append(c)
    else:
        high_cat.append(c)

if len(small_cat) > 0:
    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X_train[small_cat] = oe.fit_transform(X_train[small_cat])
    X_test[small_cat] = oe.transform(X_test[small_cat])

for c in high_cat:
    all_vals = pd.concat([X_train[c], X_test[c]], axis=0).astype(str)
    mapping = {v: i for i, v in enumerate(all_vals.value_counts().index)}
    X_train[c] = X_train[c].map(mapping).fillna(-1).astype(int)
    X_test[c] = X_test[c].map(mapping).fillna(-1).astype(int)

print('Encoded categorical columns:', len(cat_cols))

## Обучение модели

In [ ]:
te_cols = [c for c in ['location', 'location_cluster', 'type_house', 'geo_cluster_25', 'location_cluster_type_house'] if c in X_train.columns]

def make_target_bins(y):
    y = pd.Series(y)
    bins = pd.Series(index=y.index, dtype='int64')
    bins[y == 0] = 0

    non_zero = y[y > 0]
    if len(non_zero) > 10:
        q1, q2 = non_zero.quantile([0.35, 0.75])
    else:
        q1, q2 = 1, 10

    bins[(y > 0) & (y <= q1)] = 1
    bins[(y > q1) & (y <= q2)] = 2
    bins[y > q2] = 3
    return bins.fillna(1).astype(int).values


def add_oof_target_encoding(X_tr, X_val, y_tr, cols):
    X_tr = X_tr.copy()
    X_val = X_val.copy()
    global_mean = y_tr.mean()

    inner_cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED)
    y_bins = make_target_bins(y_tr)

    for col in cols:
        tr_encoded = np.zeros(len(X_tr))

        for i_tr, i_va in inner_cv.split(X_tr, y_bins):
            fold_map = pd.DataFrame({'cat': X_tr.iloc[i_tr][col], 'y': y_tr[i_tr]}).groupby('cat')['y'].mean()
            tr_encoded[i_va] = X_tr.iloc[i_va][col].map(fold_map).fillna(global_mean).values

        full_map = pd.DataFrame({'cat': X_tr[col], 'y': y_tr}).groupby('cat')['y'].mean()
        val_encoded = X_val[col].map(full_map).fillna(global_mean).values

        X_tr[f'{col}_te'] = tr_encoded
        X_val[f'{col}_te'] = val_encoded

    return X_tr, X_val


def fit_one_fold(X_tr, y_tr, X_val, y_val):
    use_lgbm = True
    model = None

    try:
        import lightgbm as lgb
        from lightgbm import LGBMRegressor

        model = LGBMRegressor(
            n_estimators=3000,
            learning_rate=0.03,
            num_leaves=64,
            max_depth=-1,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=2.0,
            objective='regression',
            metric='rmse',
            random_state=SEED,
            n_jobs=-1
        )

        callbacks = [lgb.early_stopping(150), lgb.log_evaluation(300)]
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric='rmse', callbacks=callbacks)

    except Exception as e:
        use_lgbm = False
        from catboost import CatBoostRegressor

        model = CatBoostRegressor(
            iterations=3000,
            learning_rate=0.03,
            depth=8,
            loss_function='RMSE',
            random_seed=SEED,
            verbose=300
        )
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True, verbose=300)

    return model, use_lgbm


bins = make_target_bins(y)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

oof_pred = np.zeros(len(X_train))
test_pred = np.zeros(len(X_test))
fold_scores = []

for fold, (tr_idx, val_idx) in enumerate(cv.split(X_train, bins), 1):
    X_tr, X_val = X_train.iloc[tr_idx].copy(), X_train.iloc[val_idx].copy()
    y_tr, y_val = y[tr_idx], y[val_idx]

    if len(te_cols) > 0:
        X_tr, X_val = add_oof_target_encoding(X_tr, X_val, y_tr, te_cols)

        # target encoding for test: только по train fold
        global_mean = y_tr.mean()
        X_te = X_test.copy()
        for col in te_cols:
            full_map = pd.DataFrame({'cat': X_tr[col], 'y': y_tr}).groupby('cat')['y'].mean()
            X_te[f'{col}_te'] = X_te[col].map(full_map).fillna(global_mean)
    else:
        X_te = X_test.copy()

    model, is_lgbm = fit_one_fold(X_tr, y_tr, X_val, y_val)

    val_pred = model.predict(X_val)
    te_pred = model.predict(X_te)

    oof_pred[val_idx] = val_pred
    test_pred += te_pred / cv.n_splits

    fold_mse = mean_squared_error(y_val, val_pred)
    fold_rmse = fold_mse ** 0.5
    fold_scores.append((fold_mse, fold_rmse))
    model_name = 'LightGBM' if is_lgbm else 'CatBoost'
    print(f'Fold {fold}: MSE={fold_mse:.5f}, RMSE={fold_rmse:.5f}, model={model_name}')


## Проверка качества

In [ ]:
cv_mse = mean_squared_error(y, oof_pred)
cv_rmse = cv_mse ** 0.5
print(f'Overall CV MSE: {cv_mse:.5f}')
print(f'Overall CV RMSE: {cv_rmse:.5f}')

err_df = pd.DataFrame({'target': y, 'pred': oof_pred})

def err_group(v):
    if v == 0:
        return 'zero'
    if v <= np.quantile(y[y > 0], 0.4):
        return 'low'
    if v <= np.quantile(y[y > 0], 0.8):
        return 'middle'
    return 'high'

err_df['group'] = err_df['target'].apply(err_group)

summary = err_df.groupby('group').apply(
    lambda d: pd.Series({
        'mean_target': d['target'].mean(),
        'mean_pred': d['pred'].mean(),
        'MAE': mean_absolute_error(d['target'], d['pred']),
        'MSE': mean_squared_error(d['target'], d['pred']),
        'count': len(d)
    })
).reset_index()

display(summary)

plt.figure(figsize=(6, 6))
sns.scatterplot(data=err_df.sample(min(10000, len(err_df)), random_state=SEED), x='target', y='pred', alpha=0.35, s=14)
mx = max(err_df['target'].max(), err_df['pred'].max())
plt.plot([0, mx], [0, mx], color='red', linestyle='--')
plt.title('Target vs Prediction (OOF)')
plt.show()

## Submission

In [ ]:
final_pred = np.clip(test_pred, 0, 365)

submission = sample_sub.copy()
submission['prediction'] = final_pred
submission.to_csv('submission_lgbm_alternative.csv', index=False)

display(submission.head())
display(submission['prediction'].describe())

## Outro / Идеи для улучшения
1. **Стекинг нескольких моделей** (LGBM + CatBoost + HistGBR): снизит variance и обычно стабилизирует leaderboard; реализовать через OOF-предсказания базовых моделей и мета-регрессор.
2. **Более аккуратные гео-признаки** (h3/geohash + агрегаты по ячейкам): лучше учитывают локальные микрорайоны; реализовать через дискретизацию lat/lon и вычисление статистик по ячейкам без target leakage.
3. **Тонкая оптимизация CV и hyperparams** (Optuna + robust objective): может улучшить RMSE за счёт лучшего bias/variance баланса; реализовать с ограничением по времени и фиксированным random_state для воспроизводимости.
